Տնային առաջադրանքը կատարելու վերջնաժամկետը **28.06.2021 16:00** է։ Առաջադրանքները կատարելուց հետո, պետք է 

1. ներբեռնեք այս ֆայլը ձեր համակարգիչ (`File` $\to$ `Download .ipynb`)

2. ներբեռնեք լրացված ``.py`` ֆայլերը

3. վերոնշյալ ֆայլերը ``.zip`` արեք և վերնագրեք հետևյալ ֆորմատով՝ 
    *HW6_AnunAzganun.zip* 

4. ուղարկեք ֆայլը **fast.1991.ml@gmail.com** հասցեին` նամակի թեմա (subject) դաշտում գրելով **ML6**

**Ուշադրություն**․ 
Վերոնշյալ պայմաններին չբավարարելու դեպքում ձեր աշխատանքը չի գնահատվի։

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('seaborn-whitegrid') # Plot style

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

%load_ext autoreload
%autoreload 2

Import the dataset `data.csv` from the `Lab_files/Lab6` folder and complete the `MyLinearRegression` class in `regression.py` script.

In [2]:
data = pd.read_csv("data.csv", index_col = 0)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(data.drop(columns=['y']),
                                                    data['y'], test_size=0.5,
                                                    random_state=0 )

# Regression Trees

Implement regression trees on your own, by slightly modifying the decision tree implementation for classification. You should change the impurity function and the way we aggregate the results of the leaves.



In [9]:
from regression import RegressionTree
model = RegressionTree()
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('Regression tree mse:', mean_squared_error(pred, y_test))

Regression tree mse: 161.90754572881585


# Support Vector Regression

Implement Support Vector Machine for regression (with support for kernels) in `SVR` class, you can use [`cvxpy`](https://www.cvxpy.org/) library to construct the quadratic optimization problem with two variables ($\alpha, \, \alpha'$). See [this](https://www.cvxpy.org/examples/basic/quadratic_program.html) for example.



In [5]:
from regression import SVR
model = SVR()
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('SVR mse:', mean_squared_error(pred, y_test))

SVR mse: 506.28632094711685


In [7]:
from sklearn.svm import SVR
model = SVR(kernel='linear')
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('SVR mse:', mean_squared_error(pred, y_test))

SVR mse: 18.615605166068285


# Linear Classification

Test the below implementations on the MNIST dataset using 0 and 1 digits for `logistic regression` and all digits for `softmax classifier`.

# Logistic Regression

Implement `LogisticRegression` class for Logistic Regression Classifier. Use gradient descent for optimization.

In [8]:
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score

In [9]:
#Getting ones and zeros only form the dataset
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_train = X_train[(y_train == 1) | (y_train == 0)]
y_train = y_train[(y_train == 1) | (y_train == 0)]
X_test = X_test[(y_test == 1) | (y_test == 0)]
y_test = y_test[(y_test == 1) | (y_test == 0)]

In [10]:
#Making 2D X_train and X_test 
X_train_flat = np.zeros((X_train.shape[0], 28*28))
for i in range(len(X_train)):
    X_train_flat[i] = X_train[i].ravel()

X_test_flat = np.zeros((X_test.shape[0], 28*28))
for i in range(len(X_test)):
    X_test_flat[i] = X_test[i].ravel()

In [11]:
# Normalizing Data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_flat = scaler.fit_transform(X_train_flat)
X_test_flat = scaler.fit_transform(X_test_flat)

In [12]:
#with batchsize 500 accuracy is higher
from regression import LogisticRegression
model = LogisticRegression(batch_size=500)
model.fit(X_train_flat , y_train)
pred = model.predict(X_test_flat)
accuracy_score(y_test, pred)

converged
26 iterations


0.9938534278959811

# Softmax Classifier

Implement `SoftmaxClassifier` class to perform classification with Softmax Classifier, which is the generalization of Logisitic Regression for multi-class classification case. Here are some gradient calculations that will be useful to perform gradient descent.




Softmax Classifier can be viewed as a 1-layer neural network. 

* Input to the network is a vector $\mathbf{x}$ of $n$ features, output of the network is a vector of $m$ class probablities $\mathbf{p}$. 

* The target class $c$ is coded as one-hot vector $\mathbf{y}$ (meaning it has 1 at index $c$ and zeroes everywhere else). 

* Weights of the network are represented by $n \times m$ matrix $\mathbf{W}$.

$$
\mathbf{x} = \begin{pmatrix} x_1\\ x_2\\ ..\\ x_n \end{pmatrix}
\qquad
\mathbf{p} = \begin{pmatrix} p_1\\ p_2\\ ..\\ p_m \end{pmatrix}
\qquad
\mathbf{y} = \begin{pmatrix} y_1\\ y_2\\ ..\\ y_m \end{pmatrix}
\qquad y_i = 
\begin{cases}
    1, & \text{if}\ \ i=c\\
    0, & \text{otherwise}
\end{cases}
\qquad
\mathbf{W} = \begin{pmatrix} w_{11}&w_{12}&..&w_{1m}\\w_{21}&w_{22}&..&w_{2m}\\..&..&..&..\\w_{n1}&w_{n2}&..&w_{nm} \end{pmatrix}
$$

We add another feature to the inputs, which is always $1$ and this turns weights of this feature into biases. 

To train the logistic regression model (network) we use cross-entropy loss function and perform gradient descent with respect to weight matrix $\mathbf{W}$. Following represents step-by-step forward pass of the network, where $L$ is the loss function:

$$
\begin{align*}
\mathbf{z} &= \mathbf{x}^TW,\qquad\qquad z_j = \sum_{i=1}^n x_i W_{ij}
\\
\mathbf{p} &= softmax(\mathbf{z}),\qquad p_i = \frac{e^{z_i}}{\sum_{j=1}^m e^{z_j}}
\\
L &= -\sum_{i=1}^m y_i \log p_i = -\log p_c
\end{align*}
$$

To perform gradient descent we need the gradient of loss function with respect to the weights $\frac{\partial L}{\partial W_{ij}}$. This can be expressed with chain rule:

$$
%\frac{\partial L}{\partial \mathbf{W}} = \frac{\partial L}{\partial \mathbf{p}} \cdot \frac{\partial \mathbf{p}}{\partial \mathbf{z}} \cdot \frac{\partial \mathbf{z}}{\partial \mathbf{W}}
%\\
\frac{\partial L}{\partial W_{ij}} = \sum_{k=1}^{m} \sum_{l=1}^{m} \frac{\partial L}{\partial p_l} \frac{\partial p_l}{\partial z_k} \frac{\partial z_k}{\partial W_{ij}}
$$

Sum over $l$ and $k$ comes from the fact that $W_{ij}$ affects loss $L$ through multiple pathways through different $p_l$ and $z_k$. It is best imagined as a [computational graph](http://colah.github.io/posts/2015-08-Backprop/).

To compute the full gradient we need to produce three partial derivatives:

$$
\begin{align*}
\frac{\partial L}{\partial p_l} &= -y_{l}\frac{1}{p_{l}}
\\
\frac{\partial p_l}{\partial z_k} &= 
\begin{cases}
\displaystyle -\frac{e^{z_{k}}e^{z_{l}}}{(\sum_{j}e^{z_{j}})^2}=-p_{k}p_{l} \quad \text{if} \quad k\neq l \quad\\
\displaystyle \frac{e^{z_{k}}}{\sum_{j}e^{z_{j}}}-\Big(\frac{e^{z_{k}}}{\sum_{j}e^{z_{j}}}\Big)^2=p_{k}(1-p_{k}) \quad \text{if} \quad k=l
\end{cases}
\\
\frac{\partial z_k}{\partial W_{ij}} &= 
\begin{cases}
0 \quad \text{if} \quad j \neq k \\
x_{i} \quad \text{if} \quad j = k
\end{cases}
\end{align*}
$$


Putting everything together:

$$
\begin{align*}
\frac{\partial L}{\partial z_k} &= \sum_{l=1}^m \frac{\partial L}{\partial p_l} \frac{\partial p_l}{\partial z_k}= -\frac{y_k}{p_k} p_k (1-p_k) + \sum_{l\neq k}^m \frac{y_l}{p_l} p_l p_k = -y_k + y_k p_k + \sum_{l\neq k}^m y_l p_k  = -y_k + p_k \sum_{l=1}^m y_l= p_k-y_k
\\
\frac{\partial L}{\partial W_{ij}} &= \sum_{k=1}^m \frac{\partial L}{\partial z_k} \frac{\partial z_k}{\partial W_{ij}} = (p_j - y_j) x_i
\end{align*}
$$

To perform a gradient descent you need to subtract gradient from the weights (because we are minimizing the loss function):
$$
W_{ij} = W_{ij} - \alpha \frac{\partial L}{\partial W_{ij}}
$$


In [13]:
from regression import SoftmaxClassifier
model = SoftmaxClassifier()
model.fit(X_train_flat, y_train)
pred = model.predict(X_test_flat)
print(f'Test accuracy {accuracy_score(y_test, pred)} ')
print(f'Train accuracy {accuracy_score(y_train, model.predict(X_train_flat))} ')


Model Training: 100% [------------------------------------------] Time: 0:00:00


Test accuracy 0.9947990543735225 
Train accuracy 0.9932096328464272 
